DATA PROCESSING, CLEANING, IMPUTATION, AND ANALYSIS

DATA PROCESSING

In [24]:
# 1. Installing and importing modules

!pip install pandas
!pip install geopandas 

import pandas as pd
import numpy as np
import geopandas as gpd
from datetime import datetime, timedelta
from shapely.geometry import Point

In [36]:
# 2. Loading the dataset

df = pd.read_csv(r"C:\Users\madiv\My Drive\Data Analysis Projects\Collisions Analysis\Dataset\dataset.csv", low_memory=False) 
df

,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,ON STREET NAME,CROSS STREET NAME,OFF STREET NAME,...,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
0,09/11/2021,02:39,NaN,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,...,Unspecified,NaN,NaN,NaN,4455765,Sedan,Sedan,NaN,NaN,NaN
1,03/26/2022,11:45,NaN,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,...,NaN,NaN,NaN,NaN,4513547,Sedan,NaN,NaN,NaN,NaN
2,11/01/2023,01:29,BROOKLYN,11230.0,40.621790,-73.970024,"(40.62179, -73.970024)",OCEAN PARKWAY,AVENUE K,NaN,...,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
3,06/29/2022,06:55,NaN,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4541903,Sedan,Pick-up Truck,NaN,NaN,NaN
4,09/21/2022,13:21,NaN,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,NaN,...,Unspecified,NaN,NaN,NaN,4566131,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048570,06/25/2017,22:45,BRONX,10454.0,40.811650,-73.914480,"(40.81165, -73.91448)",SAINT ANNS AVENUE,EAST 145 STREET,NaN,...,NaN,NaN,NaN,NaN,3703853,NaN,NaN,NaN,NaN,NaN
1048571,06/18/2017,06:07,QUEENS,11368.0,40.756287,-73.860725,"(40.756287, -73.860725)",108 STREET,34 AVENUE,NaN,...,Unspecified,Unspecified,NaN,NaN,3694175,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN
1048572,06/20/2017,11:00,NaN,NaN,40.738070,-73.937560,"(40.73807, -73.93756)",LONG ISLAND EXPRESSWAY,NaN,NaN,...,Unspecified,NaN,NaN,NaN,3694967,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN
1048573,06/29/2017,06:50,NaN,NaN,40.695015,-73.988760,"(40.695015, -73.98876)",ADAMS STREET,NaN,NaN,...,Unspecified,Unspecified,NaN,NaN,3701768,Station Wagon/Sport Utility Vehicle,Station Wagon/Sport Utility Vehicle,Sedan,NaN,NaN


In [37]:
# 3. Understanding the data

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 29 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   CRASH DATE                     1048575 non-null  object 
 1   CRASH TIME                     1048575 non-null  object 
 2   BOROUGH                        680753 non-null   object 
 3   ZIP CODE                       680579 non-null   float64
 4   LATITUDE                       970362 non-null   float64
 5   LONGITUDE                      970362 non-null   float64
 6   LOCATION                       970362 non-null   object 
 7   ON STREET NAME                 782655 non-null   object 
 8   CROSS STREET NAME              504064 non-null   object 
 9   OFF STREET NAME                264807 non-null   object 
 10  NUMBER OF PERSONS INJURED      1048562 non-null  float64
 11  NUMBER OF PERSONS KILLED       1048555 non-null  float64
 12  NUMBER OF PEDE

DATA CLEANING AND IMPUTATION PIPELINE

In [38]:
import re
import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry import Point


# ==========================================
# 1. INITIAL ANALYSIS & DUPLICATE REMOVAL
# ==========================================
def handle_duplicates_and_basics(df):
    print("=" * 50 + "\nSTARTING CLEANING PROCESS\n" + "=" * 50)
    initial_missing = df.isna().sum()
    print(
        f"\nInitial missing values:\n{initial_missing[initial_missing > 0]}\n"
    )

    df_clean = df.copy()
    total_duplicates = df_clean.duplicated().sum()
    print(f"Total duplicate rows: {total_duplicates}")

    if total_duplicates > 0:
        df_clean = df_clean.drop_duplicates(keep="first")
        print(f"Removed duplicates. New shape: {df_clean.shape}")

    return df_clean


# ==========================================
# 2. COLUMN LOWERCASE, CLEANING & SHORTENING
# ==========================================
def standardize_and_shorten_columns(df):
    print(
        "\n" + "=" * 50 + "\nCOLUMN STANDARDIZATION & SHORTENING\n" + "=" * 50
    )

    # Normalize column names
    df.columns = [col.lower().strip().replace(" ", "_") for col in df.columns]

    rename_dict = {
        "crash_date": "crash_date",
        "crash_time": "crash_time",
        "borough": "borough",
        "zip_code": "zip_code",
        "latitude": "latitude",
        "longitude": "longitude",
        "location": "location",
        "on_street_name": "on_street",
        "cross_street_name": "cross_street",
        "off_street_name": "off_street",
        "number_of_persons_injured": "persons_injured",
        "number_of_persons_killed": "persons_killed",
        "number_of_pedestrians_injured": "ped_injured",
        "number_of_pedestrians_killed": "ped_killed",
        "number_of_cyclist_injured": "cyclist_injured",
        "number_of_cyclist_killed": "cyclist_killed",
        "number_of_motorist_injured": "motorist_injured",
        "number_of_motorist_killed": "motorist_killed",
        "collision_id": "collision_id",
    }

    for i in range(1, 6):
        rename_dict[f"contributing_factor_vehicle_{i}"] = f"factor_{i}"
        rename_dict[f"vehicle_type_code_{i}"] = f"vehicle_{i}"

    df = df.rename(columns=rename_dict)

    print(f"Shortened columns list:\n{df.columns.tolist()}")
    return df


# ==========================================
# 3. DATE/TIME CLEANING & FEATURE EXTRACTION
# ==========================================
def clean_date_time(df):
    print("\n" + "=" * 50 + "\nDATE/TIME CLEANING\n" + "=" * 50)

    if "crash_date" in df.columns:
        df["crash_date"] = pd.to_datetime(
            df["crash_date"], format="mixed", dayfirst=True, errors="coerce"
        )
        df["day_of_week"] = df["crash_date"].dt.day_name()
        df["month"] = df["crash_date"].dt.month
        df["year"] = df["crash_date"].dt.year
        df["quarter"] = df["crash_date"].dt.quarter

    if "crash_time" in df.columns:
        time_dt = pd.to_datetime(
            df["crash_time"], format="%H:%M", errors="coerce"
        )
        df["hour"] = time_dt.dt.hour
        df["minute"] = time_dt.dt.minute
        df["crash_time"] = pd.to_timedelta(time_dt.dt.strftime("%H:%M:%S"))
        df["time_of_day"] = pd.cut(
            df["hour"],
            bins=[-1, 6, 12, 17, 21, 24],
            labels=["Late Night", "Morning", "Afternoon", "Evening", "Night"],
            right=False,
        )

    print("Date and Time features processed successfully.")
    return df


# ==========================================
# 4. TARGETED COORDINATE & SPATIAL IMPUTATION
# ==========================================
def auto_impute_from_existing_data(df):
    print("\nRunning Step 1: Internal Coordinate Imputation...")

    if "latitude" not in df.columns or "longitude" not in df.columns:
        return df

    df["lat_round"] = df["latitude"].round(5)
    df["lon_round"] = df["longitude"].round(5)

    target_cols = [
        "borough",
        "zip_code",
        "on_street",
        "cross_street",
        "off_street",
    ]
    target_cols = [c for c in target_cols if c in df.columns]

    valid_coords = df.dropna(subset=["lat_round", "lon_round"])

    if not valid_coords.empty and len(target_cols) > 0:
        lookup_table = (
            valid_coords.groupby(["lat_round", "lon_round"])[target_cols]
            .first()
            .reset_index()
        )

        df = df.merge(
            lookup_table,
            on=["lat_round", "lon_round"],
            how="left",
            suffixes=("", "_imputed"),
        )

        for col in target_cols:
            df[col] = df[col].fillna(df[f"{col}_imputed"])
            df.drop(columns=[f"{col}_imputed"], inplace=True)

    df.drop(columns=["lat_round", "lon_round"], inplace=True)
    return df


def impute_borough_spatially(df):
    print("Running Step 2: Spatial Join for remaining Boroughs...")

    missing_mask = (
        df["latitude"].notna()
        & (df["latitude"] != 0)
        & df["longitude"].notna()
        & (df["longitude"] != 0)
        & df["borough"].isna()
    )

    if not missing_mask.any():
        print("No missing boroughs left to impute spatially.")
        return df

    geometry = [
        Point(xy)
        for xy in zip(
            df.loc[missing_mask, "longitude"], df.loc[missing_mask, "latitude"]
        )
    ]
    points_gdf = gpd.GeoDataFrame(
        df[missing_mask], geometry=geometry, crs="EPSG:4326"
    )

    try:
        borough_url = "https://raw.githubusercontent.com/dwillis/nyc-maps/master/boroughs.geojson"
        boroughs_gdf = gpd.read_file(borough_url).to_crs("EPSG:4326")
        joined = gpd.sjoin(
            points_gdf, boroughs_gdf, how="left", predicate="within"
        )
        df.loc[missing_mask, "borough"] = df.loc[missing_mask, "borough"].fillna(
            joined["BoroName"].str.upper()
        )
    except Exception as e:
        print(f"Spatial join skipped due to network/resource issue: {e}")

    return df


def handle_location_fallback(df):
    if "zip_code" in df.columns and "borough" in df.columns:
        zip_to_borough = (
            df.groupby("zip_code")["borough"].first().dropna().to_dict()
        )
        df["borough"] = df["borough"].fillna(df["zip_code"].map(zip_to_borough))

    if "borough" in df.columns:
        df["borough"] = df["borough"].fillna("Unspecified")

    if "zip_code" in df.columns:
        df["zip_code"] = df["zip_code"].fillna("Unknown")

    return df


# ==========================================
# 5. GENERAL VALUE IMPUTATION & STANDARDIZATION
# ==========================================
def general_imputation(df):
    print(
        "\n" + "=" * 50 + "\nGENERAL IMPUTATION & CLEANING\n" + "=" * 50
    )

    string_cols = (
        [
            "borough",
            "on_street",
            "cross_street",
            "off_street",
            "time_of_day",
            "day_of_week",
        ]
        + [f"factor_{i}" for i in range(1, 6)]
        + [f"vehicle_{i}" for i in range(1, 6)]
    )

    for col in string_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.title()
            df[col] = df[col].replace(
                {
                    "Nan": np.nan,
                    "None": np.nan,
                    "Nyc": "New York",
                    "Unspecified": "Unspecified",
                }
            )

    primary_text_cols = [
        "borough",
        "location",
        "on_street",
        "cross_street",
        "off_street",
        "vehicle_1",
        "factor_1",
    ]
    for col in primary_text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("Unspecified")

    secondary_cols = [f"factor_{i}" for i in range(2, 6)] + [
        f"vehicle_{i}" for i in range(2, 6)
    ]
    for col in secondary_cols:
        if col in df.columns:
            df[col] = df[col].fillna("Not Applicable")

    count_cols = [
        "persons_injured",
        "persons_killed",
        "ped_injured",
        "ped_killed",
        "cyclist_injured",
        "cyclist_killed",
        "motorist_injured",
        "motorist_killed",
    ]
    for col in count_cols:
        if col in df.columns:
            df[col] = (
                pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
            )

    if "persons_injured" in df.columns:
        df["total_injured"] = df["persons_injured"]
    if "persons_killed" in df.columns:
        df["total_killed"] = df["persons_killed"]

    return df


# ==========================================
# 6. PIPELINE CONTROLLER
# ==========================================
def complete_cleaning_pipeline(df_input):
    df_clean = handle_duplicates_and_basics(df_input)
    df_clean = standardize_and_shorten_columns(df_clean)
    df_clean = clean_date_time(df_clean)

    # Integrated Spatial & Location Imputation
    df_clean = auto_impute_from_existing_data(df_clean)
    df_clean = impute_borough_spatially(df_clean)
    df_clean = handle_location_fallback(df_clean)

    # General Text/Numeric Imputation
    df_clean = general_imputation(df_clean)

    print("\n" + "=" * 50 + "\nCLEANING COMPLETE\n" + "=" * 50)
    remaining = df_clean.isna().sum().sum()
    print(
        "✓ All missing values handled!"
        if remaining == 0
        else f"Note: {remaining} minor null elements remain."
    )

    return df_clean


# ==========================================
# 7. EXECUTION (Triggers the prints)
# ==========================================
df_clean = complete_cleaning_pipeline(df)

STARTING CLEANING PROCESS

Initial missing values:
BOROUGH                           367822
ZIP CODE                          367996
LATITUDE                           78213
LONGITUDE                          78213
LOCATION                           78213
ON STREET NAME                    265920
CROSS STREET NAME                 544511
OFF STREET NAME                   783768
NUMBER OF PERSONS INJURED             13
NUMBER OF PERSONS KILLED              20
CONTRIBUTING FACTOR VEHICLE 1       4475
CONTRIBUTING FACTOR VEHICLE 2     194685
CONTRIBUTING FACTOR VEHICLE 3     965927
CONTRIBUTING FACTOR VEHICLE 4    1028595
CONTRIBUTING FACTOR VEHICLE 5    1042847
VEHICLE TYPE CODE 1                 9733
VEHICLE TYPE CODE 2               272870
VEHICLE TYPE CODE 3               970752
VEHICLE TYPE CODE 4              1029537
VEHICLE TYPE CODE 5              1043058
dtype: int64

Total duplicate rows: 0

COLUMN STANDARDIZATION & SHORTENING
Shortened columns list:
['crash_date', 'crash_time', '

In [39]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 38 columns):
 #   Column            Non-Null Count    Dtype          
---  ------            --------------    -----          
 0   crash_date        1048575 non-null  datetime64[ns] 
 1   crash_time        1048575 non-null  timedelta64[ns]
 2   borough           1048575 non-null  object         
 3   zip_code          1048575 non-null  object         
 4   latitude          970362 non-null   float64        
 5   longitude         970362 non-null   float64        
 6   location          1048575 non-null  object         
 7   on_street         1048575 non-null  object         
 8   cross_street      1048575 non-null  object         
 9   off_street        1048575 non-null  object         
 10  persons_injured   1048575 non-null  int64          
 11  persons_killed    1048575 non-null  int64          
 12  ped_injured       1048575 non-null  int64          
 13  ped_killed        1048575 n

In [40]:
df_clean

,crash_date,crash_time,borough,zip_code,latitude,longitude,location,on_street,cross_street,off_street,...,vehicle_5,day_of_week,month,year,quarter,hour,minute,time_of_day,total_injured,total_killed
0,2021-11-09,0 days 02:39:00,Unspecified,Unknown,NaN,NaN,Unspecified,Whitestone Expressway,20 Avenue,Unspecified,...,Not Applicable,Tuesday,11,2021,4,2,39,Late Night,2,0
1,2022-03-26,0 days 11:45:00,Unspecified,Unknown,NaN,NaN,Unspecified,Queensboro Bridge Upper,Unspecified,Unspecified,...,Not Applicable,Saturday,3,2022,1,11,45,Morning,1,0
2,2023-01-11,0 days 01:29:00,Brooklyn,11230.0,40.621790,-73.970024,"(40.62179, -73.970024)",Ocean Parkway,Avenue K,Ocean Parkway,...,Not Applicable,Wednesday,1,2023,1,1,29,Late Night,1,0
3,2022-06-29,0 days 06:55:00,Unspecified,Unknown,NaN,NaN,Unspecified,Throgs Neck Bridge,Unspecified,Unspecified,...,Not Applicable,Wednesday,6,2022,2,6,55,Morning,0,0
4,2022-09-21,0 days 13:21:00,Unspecified,Unknown,NaN,NaN,Unspecified,Brooklyn Bridge,Unspecified,Unspecified,...,Not Applicable,Wednesday,9,2022,3,13,21,Afternoon,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048570,2017-06-25,0 days 22:45:00,Bronx,10454.0,40.811650,-73.914480,"(40.81165, -73.91448)",Saint Anns Avenue,East 145 Street,Unspecified,...,Not Applicable,Sunday,6,2017,2,22,45,Night,1,0
1048571,2017-06-18,0 days 06:07:00,Queens,11368.0,40.756287,-73.860725,"(40.756287, -73.860725)",108 Street,34 Avenue,Unspecified,...,Not Applicable,Sunday,6,2017,2,6,7,Morning,1,0
1048572,2017-06-20,0 days 11:00:00,Queens,Unknown,40.738070,-73.937560,"(40.73807, -73.93756)",Long Island Expressway,Unspecified,Unspecified,...,Not Applicable,Tuesday,6,2017,2,11,0,Morning,0,0
1048573,2017-06-29,0 days 06:50:00,Brooklyn,11201.0,40.695015,-73.988760,"(40.695015, -73.98876)",Adams Street,Johnson Street,Unspecified,...,Not Applicable,Thursday,6,2017,2,6,50,Morning,3,0


In [41]:
# ==========================================
#  RESETTING THE TABLE'S INDEX COLUMN
# ==========================================

def resetting_index(df):
    """
    Resetting the table's index column
    """
    print("="*50)
    print("RESETTING DATAFRAME INDEX")
    print("="*50)
    
    # Reset index and drop the old index column
    df_reset = df.reset_index(drop=True)
    
    print(f"Index reset successfully!")
    print(f"New index range: 0 to {len(df_reset)-1}")
    print(f"Total Rows: {len(df_reset)}")
    print(f"Total Columns: {len(df_reset.columns)}")
    
    print("\n" + "="*50)
    print("CLEANED DATASET PREVIEW")
    print("="*50)
    print(df_reset.head())
    
    print("\n" + "="*50)
    print("CLEANING PROCESS COMPLETE!")
    print("="*50)
    print(f"✓ Total Rows: {len(df_reset):,}")
    print(f"✓ Total Columns: {len(df_reset.columns)}")
    print(f"✓ Memory Usage: {df_reset.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    return df_reset

# Apply the function to reset index
df_clean = resetting_index(df_clean)

RESETTING DATAFRAME INDEX
Index reset successfully!
New index range: 0 to 1048574
Total Rows: 1048575
Total Columns: 38

CLEANED DATASET PREVIEW
  crash_date      crash_time      borough zip_code  latitude  longitude  \
0 2021-11-09 0 days 02:39:00  Unspecified  Unknown       NaN        NaN   
1 2022-03-26 0 days 11:45:00  Unspecified  Unknown       NaN        NaN   
2 2023-01-11 0 days 01:29:00     Brooklyn  11230.0  40.62179 -73.970024   
3 2022-06-29 0 days 06:55:00  Unspecified  Unknown       NaN        NaN   
4 2022-09-21 0 days 13:21:00  Unspecified  Unknown       NaN        NaN   

                 location                on_street cross_street  \
0             Unspecified    Whitestone Expressway    20 Avenue   
1             Unspecified  Queensboro Bridge Upper  Unspecified   
2  (40.62179, -73.970024)            Ocean Parkway     Avenue K   
3             Unspecified       Throgs Neck Bridge  Unspecified   
4             Unspecified          Brooklyn Bridge  Unspecified   

 

In [42]:
# ==========================================
#  SAVING CLEANED DATAFRAME WITH BACKUP AND VERIFICATION.
# ==========================================

import os
from datetime import datetime


def save_cleaned_file(df, file_path, backup=True):
    """Save cleaned dataframe with backup and verification."""
    print("=" * 50)
    print("CREATING AND SAVING CLEANED DATASET")
    print("=" * 50)

    try:
        # Ensure file path ends with .csv
        if not file_path.endswith(".csv"):
            file_path = file_path + ".csv"

        # Create target directory if it doesn't exist
        os.makedirs(os.path.dirname(file_path), exist_ok=True)

        # Backup existing file safely
        if backup and os.path.exists(file_path):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            backup_path = file_path.replace(".csv", f"_backup_{timestamp}.csv")
            os.rename(file_path, backup_path)
            print(f"✓ Created backup: {os.path.basename(backup_path)}")

        # Save file
        df.to_csv(file_path, index=False, encoding="utf-8-sig")
        print(f"✓ File saved: {file_path}")
        print(f"✓ Rows: {len(df):,}, Columns: {len(df.columns)}")
        print(
            f"✓ File size: {os.path.getsize(file_path) / (1024 * 1024):.2f} MB"
        )

        # Verify file creation
        if os.path.exists(file_path):
            print("✅ File successfully created and verified!")
            return True
        else:
            print("❌ File creation failed verification!")
            return False

    except Exception as e:
        print(f"❌ Error: {e}")
        return False


# Specify FULL path including the filename
file_path = r"C:\Users\madiv\My Drive\Data Analysis Projects\Collisions Analysis\Cleaned\cleaned_collisions_data.csv"
save_cleaned_file(df_clean, file_path, backup=True)

CREATING AND SAVING CLEANED DATASET
✓ Created backup: cleaned_collisions_data_backup_20260801_173638.csv
✓ File saved: C:\Users\madiv\My Drive\Data Analysis Projects\Collisions Analysis\Cleaned\cleaned_collisions_data.csv
✓ Rows: 1,048,575, Columns: 38
✓ File size: 350.08 MB
✅ File successfully created and verified!


True

In [43]:
import os
import sqlite3
import pandas as pd

db_path = r"C:\Users\madiv\My Drive\Data Analysis Projects\Collisions Analysis\Cleaned\cleaned_collisions_data.db"
os.makedirs(os.path.dirname(db_path), exist_ok=True)

# 1. Reference your existing cleaned dataframe variable
df_db = df_clean.copy()

# Convert timedelta columns if any exist
for col in df_db.columns:
    if pd.api.types.is_timedelta64_dtype(df_db[col]):
        df_db[col] = df_db[col].dt.total_seconds().astype(int)

# Write to SQLite in chunks
chunk_size = 25000

with sqlite3.connect(db_path) as conn:
    for i in range(0, len(df_db), chunk_size):
        chunk = df_db.iloc[i : i + chunk_size]
        if i == 0:
            chunk.to_sql(
                "cleaned_collisions_data",
                conn,
                if_exists="replace",
                index=False,
            )
        else:
            chunk.to_sql(
                "cleaned_collisions_data",
                conn,
                if_exists="append",
                index=False,
            )
        print(f"✓ Wrote rows {i:,} to {min(i + chunk_size, len(df_db)):,}")

print(f"\n✅ Database created: {db_path}")
print(f"Size: {os.path.getsize(db_path) / (1024 * 1024):.2f} MB")

✓ Wrote rows 0 to 25,000
✓ Wrote rows 25,000 to 50,000
✓ Wrote rows 50,000 to 75,000
✓ Wrote rows 75,000 to 100,000
✓ Wrote rows 100,000 to 125,000
✓ Wrote rows 125,000 to 150,000
✓ Wrote rows 150,000 to 175,000
✓ Wrote rows 175,000 to 200,000
✓ Wrote rows 200,000 to 225,000
✓ Wrote rows 225,000 to 250,000
✓ Wrote rows 250,000 to 275,000
✓ Wrote rows 275,000 to 300,000
✓ Wrote rows 300,000 to 325,000
✓ Wrote rows 325,000 to 350,000
✓ Wrote rows 350,000 to 375,000
✓ Wrote rows 375,000 to 400,000
✓ Wrote rows 400,000 to 425,000
✓ Wrote rows 425,000 to 450,000
✓ Wrote rows 450,000 to 475,000
✓ Wrote rows 475,000 to 500,000
✓ Wrote rows 500,000 to 525,000
✓ Wrote rows 525,000 to 550,000
✓ Wrote rows 550,000 to 575,000
✓ Wrote rows 575,000 to 600,000
✓ Wrote rows 600,000 to 625,000
✓ Wrote rows 625,000 to 650,000
✓ Wrote rows 650,000 to 675,000
✓ Wrote rows 675,000 to 700,000
✓ Wrote rows 700,000 to 725,000
✓ Wrote rows 725,000 to 750,000
✓ Wrote rows 750,000 to 775,000
✓ Wrote rows 775,000